# Simulate traps


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys, os, importlib
sys.path.append( os.path.abspath(os.path.join('..')) )

import warnings
warnings.filterwarnings('ignore')

import utils.simulations as sim


In [ ]:
importlib.reload(sim)

locs = {'Arboleda': (0.00048, 0.5, 8), 'La_Margarita': (0.00048, 0.5, 8),
        'Playa': (0.00048, 0.5, 8), 'San_Juan': (0.00048, 0.5, 8), 
        'Villodas': (0.00048, 0.5, 8)}

data_path = '../data/Abundance_predictions'

for loc, vals in locs.items():
    df = sim.format_mols_data(loc, data_path)
    poisson_p, negbin_p, prop = vals

    poisson = sim.poisson_sim(poisson_p, df.copy(deep=True))

    seen_prop = prop/np.average(df.MoLS)
    negbin = sim.negbin_sim(negbin_p, seen_prop, df.copy(deep=True))

    poisson.to_csv('../data/Trap_sims/{}_poisson.csv'.format(loc), index=False)
    negbin.to_csv('../data/Trap_sims/{}_negbin.csv'.format(loc), index=False)


AttributeError: 'DataFrame' object has no attribute 'mols_avg'

In [ ]:
fig, axs = plt.subplots(2)


axs[0].plot(poisson.Datetime, poisson.weekly_mean, color='tab:blue', alpha=0.5, label='Mean')
axs[0].plot(poisson.Datetime, poisson.weekly_var, color='tab:orange', alpha=0.5, label='Variance')
axs[0].vlines(x=[poisson.Datetime.values[13], poisson.Datetime.values[-52]], ymin=-5, ymax = 25, color='tab:red', linestyle='--', alpha=0.5)
axs[0].set_ylim([-1, 23])

axs[1].plot(negbin.Datetime, negbin.weekly_mean, color='tab:blue', alpha=0.5, label='Mean')
#axs[1].plot(negbin.Datetime, 2*negbin.weekly_mean, color='tab:red', alpha=0.5, label='check')
axs[1].plot(negbin.Datetime, negbin.weekly_var, color='tab:orange', alpha=0.5, label='Variance')
axs[1].vlines(x=[poisson.Datetime.values[13], poisson.Datetime.values[-52]], ymin=-2, ymax = 60, color='tab:red', linestyle='--', alpha=0.5)
axs[1].set_ylim([-2, 45])

axs[0].set_ylabel('Poisson trap counts')
axs[0].set_title('Statistics of simulated traps')
axs[0].legend(loc='upper left')

axs[1].set_ylabel('Negative Binomial\ntrap counts')
axs[1].legend(loc='upper left')
fig.tight_layout()
fig.savefig('../output/simulated_trap_catches.png', dpi=300, bbox_inches='tight')
plt.show()